In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


df = pd.read_csv("../data/raw/creditcard.csv")

## Удаление дубликатов

In [66]:
df_dupl = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Размер после удаления дубликатов: {df.shape}")
print(f"Осталось дубликатов: {df.duplicated().sum()}")

Размер после удаления дубликатов: (283726, 31)
Осталось дубликатов: 0


## Создаем новые признаки Hour и log_Amount

In [67]:
df["Hour"] = (df["Time"] // 3600) % 24
df["log_Amount"] = np.log1p(df["Amount"])

print(f"Новые признаки и исходные:  \n {df[['Time', 'Hour', 'Amount', 'log_Amount']]}")


Новые признаки и исходные:  
             Time  Hour  Amount  log_Amount
0            0.0   0.0  149.62    5.014760
1            0.0   0.0    2.69    1.305626
2            1.0   0.0  378.66    5.939276
3            1.0   0.0  123.50    4.824306
4            2.0   0.0   69.99    4.262539
...          ...   ...     ...         ...
283721  172786.0  23.0    0.77    0.570980
283722  172787.0  23.0   24.79    3.249987
283723  172788.0  23.0   67.88    4.232366
283724  172788.0  23.0   10.00    2.397895
283725  172792.0  23.0  217.00    5.384495

[283726 rows x 4 columns]


## Обоснование удаления Time и Amount

### Time → удаляем
- Исходный признак Time содержит абсолютное время в секундах
- Абсолютное время не имеет прогностической ценности для новых данных
- Час суток может влиять на паттерны транзакций 
- Оставлять оба признака - мультиколлинеарность и избыточность


### Amount → удаляем
- Исходный Amount имеет сильную правостороннюю асимметрию (медиана ~22, max ~25000)
- Логарифмирование через log1p нормализует распределение, приближая к нормальному
- Модели машинного обучения лучше работают с симметричными распределениями
- log_Amount сохраняет всю информацию о сумме транзакции
- Оставлять оба признака избыточно и создает мультиколлинеарность

In [68]:
df = df.drop(['Time', 'Amount'], axis= 1)
print(f"Финальное количество признаков: {df.shape[1]}")
print(df.columns.tolist())

Финальное количество признаков: 31
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Hour', 'log_Amount']


## Train/Test split

In [69]:
X = df.drop('Class', axis=1)
y = df['Class']

train_X, test_X, train_y, test_y = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"\nДоля фрода в train: {train_y.mean()*100:.4f}%")
print(f"Доля фрода в test: {test_y.mean()*100:.4f}%")
print(f"Разница: {abs(train_y.mean() - test_y.mean())*100:.6f}%")


Доля фрода в train: 0.1665%
Доля фрода в test: 0.1674%
Разница: 0.000878%


## Scaling для Hour и log_Amount

In [70]:
scale_cols = ['Hour', 'log_Amount']

scaler = StandardScaler()
scaler.fit(train_X[scale_cols])

train_X_scaled = train_X.copy()
test_X_scaled = test_X.copy()

train_X_scaled[scale_cols] = scaler.transform(train_X[scale_cols])
test_X_scaled[scale_cols] = scaler.transform(test_X[scale_cols])

print(f"\nСтатистики до масштабирования (train):")
print(train_X[scale_cols].describe())
print(f"\nСтатистики после масштабирования (train):")
print(train_X_scaled[scale_cols].describe())
print(f"\nСтатистики до масштабирования (test):")
print(test_X[scale_cols].describe())
print(f"\nСтатистики после масштабирования (test):")
print(test_X_scaled[scale_cols].describe())


Статистики до масштабирования (train):
                Hour     log_Amount
count  226980.000000  226980.000000
mean       14.054370       3.156136
std         5.828895       1.656234
min         0.000000       0.000000
25%        10.000000       1.900614
50%        15.000000       3.138966
75%        19.000000       4.367547
max        23.000000       9.886216

Статистики после масштабирования (train):
               Hour    log_Amount
count  2.269800e+05  2.269800e+05
mean  -1.183886e-16  1.811574e-16
std    1.000002e+00  1.000002e+00
min   -2.411161e+00 -1.905614e+00
25%   -6.955657e-01 -7.580600e-01
50%    1.622317e-01 -1.036679e-02
75%    8.484697e-01  7.314265e-01
max    1.534708e+00  4.063491e+00

Статистики до масштабирования (test):
               Hour    log_Amount
count  56746.000000  56746.000000
mean      14.010750      3.144258
std        5.858365      1.660439
min        0.000000      0.000000
25%       10.000000      1.870263
50%       14.000000      3.117286
75%       

## Сохрание данных 

In [71]:
train_X_scaled.to_csv('../data/processed/X_train.csv', index=False)
test_X_scaled.to_csv('../data/processed/X_test.csv', index=False)
train_y.to_csv('../data/processed/y_train.csv', index=False)
test_y.to_csv('../data/processed/y_test.csv', index=False)


import joblib
joblib.dump(scaler, '../models/scaler.pkl')


['../models/scaler.pkl']

## Выводы по preprocessing

In [73]:
print(f"Исходный датасет: {df.shape[0]} записей")
print(f"После удаления дубликатов: {len(df)} записей")
print(f"Удалено дубликатов: {df_dupl}")
print(f"\nTrain set: {len(train_X_scaled)} записей ({len(train_X_scaled)/len(df)*100:.1f}%)")
print(f"Test set: {len(test_X_scaled)} записей ({len(test_X_scaled)/len(df)*100:.1f}%)")
print(f"\nВсего признаков: {train_X_scaled.shape[1]}")
print(f"  - PCA компоненты (V1-V28): 28")
print(f"  - Инженерные признаки (Hour, log_Amount): 2")
print(f"\nРаспределение классов:")
print(f"  Train - Фрод: {train_y.sum()} ({train_y.mean()*100:.4f}%)")
print(f"  Test  - Фрод: {test_y.sum()} ({test_y.mean()*100:.4f}%)")

Исходный датасет: 283726 записей
После удаления дубликатов: 283726 записей
Удалено дубликатов: 1081

Train set: 226980 записей (80.0%)
Test set: 56746 записей (20.0%)

Всего признаков: 30
  - PCA компоненты (V1-V28): 28
  - Инженерные признаки (Hour, log_Amount): 2

Распределение классов:
  Train - Фрод: 378 (0.1665%)
  Test  - Фрод: 95 (0.1674%)
